In [2]:
import os, json, sys, cv2, numpy as np, torch
from pathlib import Path

# ── 配置 ──────────────────────────────────────────────
VIDEO_DIR  = "Bronchoscopy_test"   # 视频文件夹
ANN_DIR    = "."                                 # annotation json 文件夹
OUTPUT_DIR = "./results"                         # 输出文件夹
CKPT_PATH  = "./checkpoints/reference_model/model-000600000.pth"

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# ── 找所有视频 ────────────────────────────────────────
video_files = sorted(Path(VIDEO_DIR).rglob("input_video.mp4"))
print(f"共找到 {len(video_files)} 个视频:")
for v in video_files:
    cap = cv2.VideoCapture(str(v))
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    print(f"  {v.relative_to(VIDEO_DIR)}  →  {n} 帧  {n/fps:.1f}s")

# ── 加载模型（只加载一次）────────────────────────────
sys.path.insert(0, "./alltracker")
from nets.alltracker import Net as AllTracker
model = AllTracker(seqlen=16).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print("模型加载完成")


共找到 14 个视频:
  real_seq_001_part_0_dif_1\Videos_000\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_001\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_002\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_003\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_004\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_005\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_006\video\input_video.mp4  →  24 帧  2.4s
  real_seq_001_part_0_dif_1\Videos_007\video\input_video.mp4  →  24 帧  2.4s
  stable_seq_001_part_1_dif_0\Videos_000\video\input_video.mp4  →  24 帧  2.4s
  stable_seq_001_part_1_dif_0\Videos_001\video\input_video.mp4  →  24 帧  2.4s
  stable_seq_001_part_1_dif_0\Videos_002\video\input_video.mp4  →  24 帧  2.4s
  stable_seq_001_part_1_dif_0\Videos_003\video\input_video.mp4  →  24 帧  2.4s
  stable_seq_001_part_1_dif_0\Videos_004\video\input_video.mp4  →  2

In [ ]:
import cv2
import json
import os
from pathlib import Path

# ── 配置 ────────────────────────────────────────
ANN_DIR    = "."                                 # 标注保存位置
MAX_FRAMES = 24
N_POINTS   = 5                                   # 每帧标注点数

# ── 找所有视频 ────────────────────────────────────────
video_files = sorted(Path(VIDEO_DIR).glob("*.mp4"))
print(f"共找到 {len(video_files)} 个视频:")
for v in video_files:
    print(f"  {v.name}")

# ── 标注函数 ──────────────────────────────────────────
def click_event(event, x, y, flags, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        params.append([x, y])
        print(f"  点 {len(params)}: ({x}, {y})")

def annotate_video(video_path, ann_path, n_points=5):
    seq_name = Path(video_path).stem
    
    # 已有标注则跳过
    if Path(ann_path).exists():
        print(f"\n[跳过] {seq_name}，标注文件已存在: {ann_path}")
        return

    print(f"\n{'='*50}")
    print(f"标注: {seq_name}")

    # 读视频
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        print(f"  [错误] 无法读取视频，跳过")
        return

    first_frame = frames[0]
    last_frame  = frames[-1]
    print(f"  共 {len(frames)} 帧，请标注第0帧和第{len(frames)-1}帧")

    points = {"f0": [], "fLast": [], "total_frames": len(frames)}

    # ── 标注第一帧 ────────────────────────────────────
    print(f"\n  >>> 第一帧：请点击 {n_points} 个点，完成后按任意键确认")
    cv2.imshow(f"[{seq_name}] First Frame - click {n_points} points, press any key", first_frame)
    cv2.setMouseCallback(f"[{seq_name}] First Frame - click {n_points} points, press any key",
                         click_event, points["f0"])
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print(f"  第一帧标注完成，共 {len(points['f0'])} 个点")

    # ── 标注最后一帧 ──────────────────────────────────
    print(f"\n  >>> 最后一帧：请点击对应的 {n_points} 个点，完成后按任意键确认")
    cv2.imshow(f"[{seq_name}] Last Frame - click {n_points} points, press any key", last_frame)
    cv2.setMouseCallback(f"[{seq_name}] Last Frame - click {n_points} points, press any key",
                         click_event, points["fLast"])
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print(f"  最后帧标注完成，共 {len(points['fLast'])} 个点")

    # ── 验证 & 保存 ───────────────────────────────────
    if len(points["f0"]) != len(points["fLast"]):
        print(f"  [警告] 点数不一致：第一帧 {len(points['f0'])} 个，最后帧 {len(points['fLast'])} 个，已跳过保存")
        return

    with open(ann_path, "w") as f:
        json.dump(points, f, indent=2)
    print(f"  标注已保存: {ann_path}")

# ── 批量标注 ──────────────────────────────────────────
for video_path in video_files:
    ann_path = Path(ANN_DIR) / f"annotation_{video_path.stem}.json"
    annotate_video(video_path, ann_path, n_points=N_POINTS)

print(f"\n{'='*50}")
print("全部标注完成！")

共找到 5 个视频:
  stable_seq_000_part_0_dif_0.mp4
  stable_seq_000_part_1_dif_0.mp4
  stable_seq_000_part_2_dif_0.mp4
  stable_seq_001_part_1_dif_0.mp4
  stable_seq_001_part_2_dif_0.mp4

标注: stable_seq_000_part_0_dif_0
  共 50 帧，请标注第0帧和第49帧

  >>> 第一帧：请点击 5 个点，完成后按任意键确认
  点 1: (268, 131)
  点 2: (353, 173)
  点 3: (182, 170)
  点 4: (373, 265)
  点 5: (276, 215)
  第一帧标注完成，共 5 个点

  >>> 最后一帧：请点击对应的 5 个点，完成后按任意键确认
  点 1: (264, 107)
  点 2: (367, 164)
  点 3: (168, 146)
  点 4: (388, 281)
  点 5: (263, 206)
  最后帧标注完成，共 5 个点
  标注已保存: annotation_stable_seq_000_part_0_dif_0.json

标注: stable_seq_000_part_1_dif_0
  共 50 帧，请标注第0帧和第49帧

  >>> 第一帧：请点击 5 个点，完成后按任意键确认
  点 1: (17, 209)
  点 2: (108, 147)
  点 3: (208, 189)
  点 4: (208, 282)
  点 5: (252, 444)
  第一帧标注完成，共 5 个点

  >>> 最后一帧：请点击对应的 5 个点，完成后按任意键确认
  点 1: (62, 121)
  点 2: (174, 94)
  点 3: (14, 195)
  点 4: (269, 163)
  点 5: (112, 460)
  最后帧标注完成，共 5 个点
  标注已保存: annotation_stable_seq_000_part_1_dif_0.json

标注: stable_seq_000_part_2_dif_0
  共 50 帧，请标注第0帧和第49帧

In [16]:
import cv2
import os
import numpy as np

def save_tracking_video(video_tensor, trajs, confs, output_path, fps=10, conf_thr=0.1):
    """
    将追踪结果渲染成视频
    video_tensor: [1, T, C, H, W]
    trajs:        [1, T, N, 2]
    confs:        [1, T, N]
    """
    B, T, C, H, W = video_tensor.shape
    N = trajs.shape[2]

    # 每个点分配一个固定颜色
    colors = [
        (0, 0, 255),    # 红
        (0, 255, 0),    # 绿
        (255, 0, 0),    # 蓝
        (0, 255, 255),  # 黄
        (255, 0, 255),  # 紫
    ]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    trajs_np = trajs[0].numpy()   # [T, N, 2]
    confs_np = confs[0].numpy()   # [T, N]

    for t in range(T):
        # 取当前帧图像
        frame = video_tensor[0, t].permute(1, 2, 0).cpu().numpy().astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        for i in range(N):
            color = colors[i % len(colors)]
            conf  = confs_np[t, i]
            px    = int(round(trajs_np[t, i, 0]))
            py    = int(round(trajs_np[t, i, 1]))

            # 越界跳过
            if not (0 <= px < W and 0 <= py < H):
                continue

            if conf > conf_thr:
                # 画当前点
                cv2.circle(frame, (px, py), 5, color, -1)
                # 画轨迹线（往前追溯 trail_len 帧）
                trail_len = 20
                for t_prev in range(max(0, t - trail_len), t):
                    if confs_np[t_prev, i] <= conf_thr:
                        continue
                    px_prev = int(round(trajs_np[t_prev, i, 0]))
                    py_prev = int(round(trajs_np[t_prev, i, 1]))
                    if not (0 <= px_prev < W and 0 <= py_prev < H):
                        continue
                    # 越老的轨迹越透明（颜色变暗）
                    alpha = (t_prev - (t - trail_len)) / trail_len
                    faded = tuple(int(c * alpha) for c in color)
                    cv2.line(frame, (px_prev, py_prev), (px, py), faded, 1)

                # 点编号标注
                cv2.putText(frame, f"P{i+1}", (px+6, py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            else:
                # 不可见：画空心圆表示追踪失效
                cv2.circle(frame, (px, py), 5, color, 1)
                cv2.putText(frame, f"P{i+1}?", (px+6, py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (128,128,128), 1)

        # 帧号
        cv2.putText(frame, f"Frame {t+1}/{T}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        writer.write(frame)

    writer.release()
    print(f"✅ 视频已保存 → {output_path}")




In [ ]:

# ── 批量处理 ──────────────────────────────────────────
all_results = {}

for video_path in video_files:
    seq_name = video_path.stem                        # e.g. "dynamic_seq_003"
    ann_path = Path(ANN_DIR) / f"annotation_{seq_name}.json"

    # 没有对应标注就跳过
    if not ann_path.exists():
        print(f"\n[跳过] {seq_name}，找不到标注文件 {ann_path}")
        continue

    print(f"\n{'='*50}")
    print(f"处理: {seq_name}")

    # 读视频
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while len(frames) < 50:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    video_tensor = torch.tensor(np.array(frames)).permute(0,3,1,2).unsqueeze(0).float().to(device)
    T = video_tensor.shape[1]
    H, W = video_tensor.shape[-2], video_tensor.shape[-1]
    print(f"  帧数={T}, 分辨率={H}x{W}")

    # 读标注
    with open(ann_path) as f:
        ann = json.load(f)
    f0_pts  = np.array(ann["f0"])
    gt_last = np.array(ann["fLast"])
    N = len(f0_pts)

    # 滑窗推理
    cur_pts   = f0_pts.copy().astype(float)
    all_trajs = np.zeros((T, N, 2))
    all_trajs[0] = cur_pts

    flow_fields = []   # ← 新增

    for start in starts:
        end  = min(start + window_size, T)
        win  = video_tensor[:, start:end]
        with torch.no_grad():
            out = model(win)
        flow = out[0]  # [1, wT, 2, fH, fW]

        flow_fields.append((start, flow[0].cpu().numpy()))  # ← 新增收集

    window_size, stride = 16, 8
    starts = list(range(0, T - window_size, stride))
    if not starts or starts[-1] + window_size < T:
        starts.append(max(0, T - window_size))

    for start in starts:
        end  = min(start + window_size, T)
        win  = video_tensor[:, start:end]
        with torch.no_grad():
            out = model(win)
        flow = out[0]
        fH, fW = flow.shape[-2], flow.shape[-1]
        scale_x, scale_y = fW / W, fH / H

        for i, (x, y) in enumerate(cur_pts):
            xi = int(round(x * scale_x)); xi = min(max(xi, 0), fW-1)
            yi = int(round(y * scale_y)); yi = min(max(yi, 0), fH-1)
            for t in range(end - start):
                dx = float(flow[0, t, 0, yi, xi]) / scale_x
                dy = float(flow[0, t, 1, yi, xi]) / scale_y
                if start + t < T:
                    all_trajs[start+t, i] = [x+dx, y+dy]

        last_t = min(end-1, T-1) - start
        for i in range(N):
            cur_pts[i] = all_trajs[start + last_t, i]

    # 计算指标
    pred_last = all_trajs[-1]
    errors = np.linalg.norm(pred_last - gt_last, axis=1)
    ATE    = errors.mean()
    print(f"  逐点误差: {errors.round(1)}")
    print(f"  平均ATE: {ATE:.1f}px")
    trajs = torch.tensor(all_trajs).unsqueeze(0).float()   # [1, T, N, 2]
    confs = torch.ones(1, T, N)                             # [1, T, N] 全1（无置信度输出）

    # ── 保存追踪视频 ──────────────────────────────────
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    video_out_path = str(Path(OUTPUT_DIR) / f"tracking_{seq_name}.mp4")
    save_tracking_video(
        video_tensor=video_tensor.cpu(),
        trajs=trajs,
        confs=confs,
        output_path=video_out_path,
        fps=10,
        conf_thr=0.1
    )

    # 保存结果
    result = {
        "seq": seq_name,
        "ATE": float(ATE),
        "per_point_error": errors.tolist(),
        "pred_last": pred_last.tolist(),
        "gt_last":   gt_last.tolist(),
    }
    all_results[seq_name] = result

    out_json = Path(OUTPUT_DIR) / f"result_{seq_name}.json"
    with open(out_json, "w") as f:
        json.dump(result, f, indent=2)

# ── 汇总 ──────────────────────────────────────────────
print(f"\n{'='*50}")
print("汇总结果:")
all_ATEs = []
for seq, res in all_results.items():
    print(f"  {seq}: ATE = {res['ATE']:.1f}px")
    all_ATEs.append(res['ATE'])

print(f"\n全部序列平均 ATE: {np.mean(all_ATEs):.1f}px")

# 保存汇总
summary = {"sequences": all_results, "mean_ATE": float(np.mean(all_ATEs))}
with open(Path(OUTPUT_DIR) / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"结果已保存到 {OUTPUT_DIR}/")



处理: stable_seq_000_part_0_dif_0
  帧数=50, 分辨率=480x480
  逐点误差: [39.9 41.9 70.9 56.2  4.5]
  平均ATE: 42.7px
✅ 视频已保存 → results\tracking_stable_seq_000_part_0_dif_0.mp4

处理: stable_seq_000_part_1_dif_0
  帧数=50, 分辨率=480x480
  逐点误差: [ 78.8 112.6 413.5 204.9 166.3]
  平均ATE: 195.2px
✅ 视频已保存 → results\tracking_stable_seq_000_part_1_dif_0.mp4

处理: stable_seq_000_part_2_dif_0
  帧数=50, 分辨率=480x480
  逐点误差: [105.5  62.8 122.   24.8  63.5]
  平均ATE: 75.7px
✅ 视频已保存 → results\tracking_stable_seq_000_part_2_dif_0.mp4

处理: stable_seq_001_part_1_dif_0
  帧数=50, 分辨率=480x480
  逐点误差: [284.8 288.4 181.3 188.1  66.1]
  平均ATE: 201.7px
✅ 视频已保存 → results\tracking_stable_seq_001_part_1_dif_0.mp4

处理: stable_seq_001_part_2_dif_0
  帧数=50, 分辨率=480x480
  逐点误差: [ 24.9  53.5 117.3 150.5 330.9]
  平均ATE: 135.4px
✅ 视频已保存 → results\tracking_stable_seq_001_part_2_dif_0.mp4

汇总结果:
  stable_seq_000_part_0_dif_0: ATE = 42.7px
  stable_seq_000_part_1_dif_0: ATE = 195.2px
  stable_seq_000_part_2_dif_0: ATE = 75.7px
  stable_seq_001_

In [18]:
import cv2, numpy as np, torch
from pathlib import Path

def flow_to_rgb(flow_xy):
    """
    flow_xy: [2, H, W] numpy, flow_xy[0]=dx, flow_xy[1]=dy
    返回 BGR 彩色光流图
    """
    dx = flow_xy[0]
    dy = flow_xy[1]

    magnitude = np.sqrt(dx**2 + dy**2)
    angle     = np.arctan2(dy, dx)  # -pi ~ pi

    # 归一化 magnitude
    mag_norm = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # HSV: H=方向, S=255, V=大小
    hsv = np.zeros((flow_xy.shape[1], flow_xy.shape[2], 3), dtype=np.uint8)
    hsv[..., 0] = ((angle + np.pi) / (2 * np.pi) * 179).astype(np.uint8)  # H: 0~179
    hsv[..., 1] = 255
    hsv[..., 2] = mag_norm

    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


def save_flow_tracking_video(video_tensor, all_trajs, flow_fields, output_path, fps=10, alpha=0.5):
    """
    video_tensor : [1, T, C, H, W]
    all_trajs    : [T, N, 2] numpy
    flow_fields  : list of (start, flow_np) where flow_np=[wT, 2, fH, fW]
    alpha        : 光流叠加透明度
    """
    B, T, C, H, W = video_tensor.shape
    N = all_trajs.shape[1]

    colors = [
        (0,   0,   255),
        (0,   255, 0),
        (255, 0,   0),
        (0,   255, 255),
        (255, 0,   255),
    ]

    # 预先把每帧对应的光流图缓存好
    frame_flow_rgb = [None] * T
    for (start, flow_np) in flow_fields:
        fH, fW = flow_np.shape[-2], flow_np.shape[-1]
        for t_local in range(flow_np.shape[0]):
            abs_t = start + t_local
            if abs_t >= T: continue
            f_rgb = flow_to_rgb(flow_np[t_local])          # [fH, fW, 3]
            f_rgb = cv2.resize(f_rgb, (W, H))              # 还原到原始分辨率
            frame_flow_rgb[abs_t] = f_rgb

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (W, H))

    for t in range(T):
        frame = video_tensor[0, t].permute(1,2,0).cpu().numpy().astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        # 叠加光流色图
        if frame_flow_rgb[t] is not None:
            frame = cv2.addWeighted(frame, 1 - alpha, frame_flow_rgb[t], alpha, 0)

        # 画轨迹和点
        trail_len = 20
        for i in range(N):
            color = colors[i % len(colors)]
            px = int(round(all_trajs[t, i, 0]))
            py = int(round(all_trajs[t, i, 1]))
            if not (0 <= px < W and 0 <= py < H): continue

            # 轨迹线
            for t2 in range(max(0, t - trail_len), t):
                px2 = int(round(all_trajs[t2, i, 0]))
                py2 = int(round(all_trajs[t2, i, 1]))
                if not (0 <= px2 < W and 0 <= py2 < H): continue
                alpha_trail = (t2 - (t - trail_len)) / trail_len
                faded = tuple(int(c * alpha_trail) for c in color)
                cv2.line(frame, (px2, py2), (px, py), faded, 1)

            cv2.circle(frame, (px, py), 6, color, -1)
            cv2.putText(frame, f"P{i+1}", (px+8, py-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        # 帧号
        cv2.putText(frame, f"Frame {t+1}/{T}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        writer.write(frame)

    writer.release()
    print(f"✅ 光流追踪视频已保存 → {output_path}")